In [1]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
drive_root = "/content/drive/MyDrive"

pos_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [5]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [6]:
train_pos_df = pd.read_csv(pos_train_csv)
val_pos_df   = pd.read_csv(pos_val_csv)
test_pos_df  = pd.read_csv(pos_test_csv)

print("Train:", train_pos_df.shape)
print("Val  :", val_pos_df.shape)
print("Test :", test_pos_df.shape)

print(train_pos_df.head())
print(train_pos_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                  unit2_pos  unit1_beam
0   3532    [0.8092883966431671, 0.521083920903955]          17
1   2224  [0.4816276084988933, 0.29434536152734486]          14
2   9416    [0.220278556834608, 0.4136596156292844]          17
3   8510  [0.21412273613497904, 0.4547214157104936]          20
4   6877  [0.14500641727379412, 0.4097884695072434]          17
['index', 'unit2_pos', 'unit1_beam']


In [7]:
label_col = train_pos_df.columns[-1]
feature_cols = [c for c in train_pos_df.columns if c != label_col]

print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

Feature columns: ['index', 'unit2_pos']
Label column: unit1_beam
Label min/max: 2 30


In [31]:
import ast

def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

for df in [train_pos_df, val_pos_df, test_pos_df]:
    parsed = df["unit2_pos"].apply(parse_unit2_pos)
    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

feature_cols = ["pos_x", "pos_y"]
label_col = "unit1_beam"

print(train_pos_df[["index", "unit2_pos", "pos_x", "pos_y", label_col]].head())
print("Feature columns:", feature_cols)
print("Label column:", label_col)
print("Original label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())
print("Unique labels:", sorted(train_pos_df[label_col].unique()))

   index                                  unit2_pos     pos_x     pos_y  \
0   3532    [0.8092883966431671, 0.521083920903955]  0.809288  0.521084   
1   2224  [0.4816276084988933, 0.29434536152734486]  0.481628  0.294345   
2   9416    [0.220278556834608, 0.4136596156292844]  0.220279  0.413660   
3   8510  [0.21412273613497904, 0.4547214157104936]  0.214123  0.454721   
4   6877  [0.14500641727379412, 0.4097884695072434]  0.145006  0.409788   

   unit1_beam  
0          17  
1          14  
2          17  
3          20  
4          17  
Feature columns: ['pos_x', 'pos_y']
Label column: unit1_beam
Original label min/max: 2 30
Unique labels: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.in

label remapping 

In [32]:
all_labels = sorted(train_pos_df[label_col].unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes_pos = len(all_labels)

print("Number of position classes:", num_classes_pos)
print("Label mapping:", label_to_id)

Number of position classes: 29
Label mapping: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3, np.int64(6): 4, np.int64(7): 5, np.int64(8): 6, np.int64(9): 7, np.int64(10): 8, np.int64(11): 9, np.int64(12): 10, np.int64(13): 11, np.int64(14): 12, np.int64(15): 13, np.int64(16): 14, np.int64(17): 15, np.int64(18): 16, np.int64(19): 17, np.int64(20): 18, np.int64(21): 19, np.int64(22): 20, np.int64(23): 21, np.int64(24): 22, np.int64(25): 23, np.int64(26): 24, np.int64(27): 25, np.int64(28): 26, np.int64(29): 27, np.int64(30): 28}


morm- only pos x,y

In [34]:
train_mean = train_pos_df[feature_cols].astype(float).mean()
train_std  = train_pos_df[feature_cols].astype(float).std().replace(0, 1)

print("Train mean:")
display(train_mean)

print("Train std:")
display(train_std)

Train mean:


,0
pos_x,0.343933
pos_y,0.396131


Train std:


,0
pos_x,0.153132
pos_y,0.115162


correct dataset 

In [35]:
class PositionBeamDataset(Dataset):
    def __init__(self, df, feature_cols, label_col, mean, std, label_to_id):
        self.df = df.reset_index(drop=True).copy()
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.mean = mean
        self.std = std
        self.label_to_id = label_to_id

        x = self.df[self.feature_cols].astype(float)
        x = (x - self.mean) / self.std

        y_raw = self.df[self.label_col].values
        y = [self.label_to_id[int(v)] for v in y_raw]

        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

correct dataloader 

In [36]:
batch_size = 64

dataset_pos_train = PositionBeamDataset(train_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_pos_val   = PositionBeamDataset(val_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_pos_test  = PositionBeamDataset(test_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)

train_loader_pos = DataLoader(dataset_pos_train, batch_size=batch_size, shuffle=True)
val_loader_pos   = DataLoader(dataset_pos_val, batch_size=batch_size, shuffle=False)
test_loader_pos  = DataLoader(dataset_pos_test, batch_size=batch_size, shuffle=False)

x_batch, y_batch = next(iter(train_loader_pos))

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First x:", x_batch[:5])
print("First mapped y:", y_batch[:10])

input_dim = x_batch.shape[1]
num_classes = num_classes_pos

print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch shape: torch.Size([64, 2])
y_batch shape: torch.Size([64])
First x: tensor([[-0.3121,  0.7657],
        [ 0.1236,  0.5220],
        [-0.8429,  0.0610],
        [-0.9941,  0.3647],
        [-0.3093,  0.8317]])
First mapped y: tensor([15, 15, 14, 16, 17, 12, 13, 17, 15, 13])
input_dim: 2
num_classes: 29


In [15]:
class PositionBeamDataset(Dataset):
    def __init__(self, csv_path, feature_cols, label_col, mean, std):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.feature_cols = feature_cols
        self.label_col = label_col

        # Keep only features with valid normalization stats (numeric columns).
        valid_cols = [c for c in mean.index if c in self.feature_cols and pd.notna(mean[c])]
        self.mean = mean[valid_cols]
        self.std = std[valid_cols].replace(0, 1).fillna(1)

        # Safely coerce string/object columns to numeric; invalid values become NaN.
        x = self.df[valid_cols].apply(pd.to_numeric, errors="coerce")
        x = x.fillna(self.mean)
        x = (x - self.mean) / self.std

        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(self.df[self.label_col].values, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [16]:
batch_size = 64

dataset_pos_train = PositionBeamDataset(pos_train_csv, feature_cols, label_col, train_mean, train_std)
dataset_pos_val   = PositionBeamDataset(pos_val_csv, feature_cols, label_col, train_mean, train_std)
dataset_pos_test  = PositionBeamDataset(pos_test_csv, feature_cols, label_col, train_mean, train_std)

train_loader_pos = DataLoader(dataset_pos_train, batch_size=batch_size, shuffle=True)
val_loader_pos   = DataLoader(dataset_pos_val, batch_size=batch_size, shuffle=False)
test_loader_pos  = DataLoader(dataset_pos_test, batch_size=batch_size, shuffle=False)

print("Train size:", len(dataset_pos_train))
print("Val size  :", len(dataset_pos_val))
print("Test size :", len(dataset_pos_test))

Train size: 6832
Val size  : 3416
Test size : 1139


In [37]:
x_batch, y_batch = next(iter(train_loader_pos))

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First x:", x_batch[:5])
print("First y:", y_batch[:10])

input_dim = x_batch.shape[1]
num_classes = 64

print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch shape: torch.Size([64, 2])
y_batch shape: torch.Size([64])
First x: tensor([[ 1.1643,  1.3479],
        [-1.0726, -0.1035],
        [ 0.5607,  0.5988],
        [ 0.2485, -0.3196],
        [ 1.0424, -2.6126]])
First y: tensor([17, 13, 15, 13,  7, 17, 16, 13, 15, 15])
input_dim: 2
num_classes: 64


4 layer mlp 

In [38]:
class FourLayerMLP(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dims=(128, 256, 128, 64), dropout=0.2):
        super().__init__()
        h1, h2, h3, h4 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h3, h4),
            nn.BatchNorm1d(h4),
            nn.ReLU(),

            nn.Linear(h4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann 

In [39]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann + attention

In [40]:
class ANNWithAttention(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        weights = self.attn(x)
        x_weighted = x * weights
        return self.classifier(x_weighted)

rnn 

In [41]:
class PositionRNN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, h = self.rnn(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

lstm 

In [42]:
class PositionLSTM(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, (h, c) = self.lstm(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

top - k evaluation

In [43]:
def evaluate_topk_pos(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device)
            labels = labels.to(device)

            outputs = model(x)

            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

trainer 

In [44]:
def train_position_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    milestones=(20, 35),
    save_path="/content/drive/MyDrive/best_position_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for x, labels in train_loader:
            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk_pos(model, val_loader, device, ks=(1, 2, 3, 5))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

model builder 

In [45]:
def build_position_model(model_name, input_dim, num_classes=num_classes_pos):
    if model_name == "mlp4":
        return FourLayerMLP(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann":
        return SimpleANN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann_attention":
        return ANNWithAttention(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "rnn":
        return PositionRNN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "lstm":
        return PositionLSTM(input_dim=input_dim, num_classes=num_classes).to(device)

    else:
        raise ValueError("Unknown model_name")

run all the models 

In [48]:
position_model_names = [
    "mlp4",
    "ann",
    "ann_attention",
    "rnn",
    "lstm"
]

all_position_histories = {}
pos_results = []

for model_name in position_model_names:
    print("\n" + "="*80)
    print(f"Training Position Model: {model_name}")
    print("="*80)

    set_seed(42)

    model_pos = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    save_path = f"/content/drive/MyDrive/best_position_{model_name}.pth"

    history_pos = train_position_model(
        model=model_pos,
        train_loader=train_loader_pos,
        val_loader=val_loader_pos,
        device=device,
        epochs=100,
        lr=1e-3,
        weight_decay=1e-4,
        milestones=(20, 35),
        save_path=save_path
    )

    all_position_histories[model_name] = history_pos

    model_pos_eval = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    model_pos_eval.load_state_dict(torch.load(save_path, map_location=device))

    test_metrics_pos = evaluate_topk_pos(model_pos_eval, test_loader_pos, device, ks=(1, 2, 3, 5))

    print(f"{model_name} Test metrics:", test_metrics_pos)

    pos_results.append({
        "Model": model_name,
        "Top-1": test_metrics_pos["top1"],
        "Top-2": test_metrics_pos["top2"],
        "Top-3": test_metrics_pos["top3"],
        "Top-5": test_metrics_pos["top5"],
    })

    del model_pos
    del model_pos_eval
    torch.cuda.empty_cache()

pos_results_df = pd.DataFrame(pos_results)
pos_results_df


Training Position Model: mlp4
Epoch 01 | Train Loss: 2.2466 | Top1: 48.39 | Top2: 71.49 | Top3: 83.23 | Top5: 91.80
Saved best model
Epoch 02 | Train Loss: 1.6180 | Top1: 49.21 | Top2: 75.29 | Top3: 85.92 | Top5: 93.85
Saved best model
Epoch 03 | Train Loss: 1.5377 | Top1: 51.67 | Top2: 75.91 | Top3: 86.65 | Top5: 94.29
Saved best model
Epoch 04 | Train Loss: 1.4970 | Top1: 50.06 | Top2: 75.94 | Top3: 87.85 | Top5: 94.88
Epoch 05 | Train Loss: 1.4716 | Top1: 54.95 | Top2: 76.99 | Top3: 87.65 | Top5: 95.52
Saved best model
Epoch 06 | Train Loss: 1.4819 | Top1: 52.75 | Top2: 75.67 | Top3: 86.21 | Top5: 95.02
Epoch 07 | Train Loss: 1.4382 | Top1: 53.83 | Top2: 77.63 | Top3: 87.47 | Top5: 95.49
Epoch 08 | Train Loss: 1.4343 | Top1: 54.51 | Top2: 77.08 | Top3: 87.85 | Top5: 95.61
Epoch 09 | Train Loss: 1.4499 | Top1: 53.31 | Top2: 76.61 | Top3: 87.35 | Top5: 95.17
Epoch 10 | Train Loss: 1.4464 | Top1: 56.26 | Top2: 78.34 | Top3: 88.50 | Top5: 95.78
Saved best model
Epoch 11 | Train Loss: 1

,Model,Top-1,Top-2,Top-3,Top-5
0,mlp4,56.189640,77.963126,89.464442,96.927129
1,ann,56.189640,81.123793,90.693591,97.190518
2,ann_attention,55.750658,79.894644,90.079017,96.927129
3,rnn,58.384548,81.035996,90.517998,97.453907
4,lstm,58.560140,82.089552,90.869183,97.541703


In [50]:
pos_results_df = pd.DataFrame(pos_results)
pos_results_df = pos_results_df.sort_values(by="Top-1", ascending=False).reset_index(drop=True)
pos_results_df

,Model,Top-1,Top-2,Top-3,Top-5
0,lstm,58.560140,82.089552,90.869183,97.541703
1,rnn,58.384548,81.035996,90.517998,97.453907
2,mlp4,56.189640,77.963126,89.464442,96.927129
3,ann,56.189640,81.123793,90.693591,97.190518
4,ann_attention,55.750658,79.894644,90.079017,96.927129


In [51]:
print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Feature sample:")
print(train_pos_df[feature_cols].head())

print("Label sample:")
print(train_pos_df[label_col].head())

print("Label unique count:", train_pos_df[label_col].nunique())
print("Label min:", train_pos_df[label_col].min())
print("Label max:", train_pos_df[label_col].max())

Feature columns: ['pos_x', 'pos_y']
Label column: unit1_beam
Feature sample:
      pos_x     pos_y
0  0.809288  0.521084
1  0.481628  0.294345
2  0.220279  0.413660
3  0.214123  0.454721
4  0.145006  0.409788
Label sample:
0    17
1    14
2    17
3    20
4    17
Name: unit1_beam, dtype: int64
Label unique count: 29
Label min: 2
Label max: 30
